In [93]:
import pandas as pd
import re
import string

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

In [94]:
# Read the airline sentiment CSV file

df = pd.read_csv('/content/Airlines Sentiments.csv')

In [95]:
df.head()

,tweet_id,airline_sentiment,airline_sentiment_confidence,negativereason,negativereason_confidence,airline,airline_sentiment_gold,name,negativereason_gold,retweet_count,text,tweet_coord,tweet_created,tweet_location,user_timezone
0,570306133677760513,neutral,1.0000,NaN,NaN,Virgin America,NaN,cairdin,NaN,0,@VirginAmerica What @dhepburn said.,NaN,2015-02-24 11:35:52 -0800,NaN,Eastern Time (US & Canada)
1,570301130888122368,positive,0.3486,NaN,0.0000,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica plus you've added commercials t...,NaN,2015-02-24 11:15:59 -0800,NaN,Pacific Time (US & Canada)
2,570301083672813571,neutral,0.6837,NaN,NaN,Virgin America,NaN,yvonnalynn,NaN,0,@VirginAmerica I didn't today... Must mean I n...,NaN,2015-02-24 11:15:48 -0800,Lets Play,Central Time (US & Canada)
3,570301031407624196,negative,1.0000,Bad Flight,0.7033,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica it's really aggressive to blast...,NaN,2015-02-24 11:15:36 -0800,NaN,Pacific Time (US & Canada)
4,570300817074462722,negative,1.0000,Can't Tell,1.0000,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica and it's a really big bad thing...,NaN,2015-02-24 11:14:45 -0800,NaN,Pacific Time (US & Canada)


In [96]:
df.shape

(14640, 15)

In [97]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14640 entries, 0 to 14639
Data columns (total 15 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   tweet_id                      14640 non-null  int64  
 1   airline_sentiment             14640 non-null  object 
 2   airline_sentiment_confidence  14640 non-null  float64
 3   negativereason                9178 non-null   object 
 4   negativereason_confidence     10522 non-null  float64
 5   airline                       14640 non-null  object 
 6   airline_sentiment_gold        40 non-null     object 
 7   name                          14640 non-null  object 
 8   negativereason_gold           32 non-null     object 
 9   retweet_count                 14640 non-null  int64  
 10  text                          14640 non-null  object 
 11  tweet_coord                   1019 non-null   object 
 12  tweet_created                 14640 non-null  object 
 13  t

In [98]:
df.describe()

,tweet_id,airline_sentiment_confidence,negativereason_confidence,retweet_count
count,1.464000e+04,14640.000000,10522.000000,14640.000000
mean,5.692184e+17,0.900169,0.638298,0.082650
std,7.791112e+14,0.162830,0.330440,0.745778
min,5.675883e+17,0.335000,0.000000,0.000000
25%,5.685592e+17,0.692300,0.360600,0.000000
50%,5.694779e+17,1.000000,0.670600,0.000000
75%,5.698905e+17,1.000000,1.000000,0.000000
max,5.703106e+17,1.000000,1.000000,44.000000


In [99]:
df.isnull().sum()

,0
tweet_id,0
airline_sentiment,0
airline_sentiment_confidence,0
negativereason,5462
negativereason_confidence,4118
airline,0
airline_sentiment_gold,14600
name,0
negativereason_gold,14608
retweet_count,0


In [100]:
df=df.drop(columns=['negativereason', 'user_timezone', 'tweet_location','tweet_coord','negativereason_gold','airline_sentiment_gold','negativereason_confidence'])

In [101]:
df

,tweet_id,airline_sentiment,airline_sentiment_confidence,airline,name,retweet_count,text,tweet_created
0,570306133677760513,neutral,1.0000,Virgin America,cairdin,0,@VirginAmerica What @dhepburn said.,2015-02-24 11:35:52 -0800
1,570301130888122368,positive,0.3486,Virgin America,jnardino,0,@VirginAmerica plus you've added commercials t...,2015-02-24 11:15:59 -0800
2,570301083672813571,neutral,0.6837,Virgin America,yvonnalynn,0,@VirginAmerica I didn't today... Must mean I n...,2015-02-24 11:15:48 -0800
3,570301031407624196,negative,1.0000,Virgin America,jnardino,0,@VirginAmerica it's really aggressive to blast...,2015-02-24 11:15:36 -0800
4,570300817074462722,negative,1.0000,Virgin America,jnardino,0,@VirginAmerica and it's a really big bad thing...,2015-02-24 11:14:45 -0800
...,...,...,...,...,...,...,...,...
14635,569587686496825344,positive,0.3487,American,KristenReenders,0,@AmericanAir thank you we got on a different f...,2015-02-22 12:01:01 -0800
14636,569587371693355008,negative,1.0000,American,itsropes,0,@AmericanAir leaving over 20 minutes Late Flig...,2015-02-22 11:59:46 -0800
14637,569587242672398336,neutral,1.0000,American,sanyabun,0,@AmericanAir Please bring American Airlines to...,2015-02-22 11:59:15 -0800
14638,569587188687634433,negative,1.0000,American,SraJackson,0,"@AmericanAir you have my money, you change my ...",2015-02-22 11:59:02 -0800


In [102]:
df.shape

(14640, 8)

In [103]:
df[['text', 'airline_sentiment']].head()

,text,airline_sentiment
0,@VirginAmerica What @dhepburn said.,neutral
1,@VirginAmerica plus you've added commercials t...,positive
2,@VirginAmerica I didn't today... Must mean I n...,neutral
3,@VirginAmerica it's really aggressive to blast...,negative
4,@VirginAmerica and it's a really big bad thing...,negative


In [104]:
text_column = df['text']
label_column = df['airline_sentiment']

In [105]:
# We only need the tweet text and sentiment label

df = df[['text', 'airline_sentiment']]

df.head()

,text,airline_sentiment
0,@VirginAmerica What @dhepburn said.,neutral
1,@VirginAmerica plus you've added commercials t...,positive
2,@VirginAmerica I didn't today... Must mean I n...,neutral
3,@VirginAmerica it's really aggressive to blast...,negative
4,@VirginAmerica and it's a really big bad thing...,negative


In [106]:
# Convert all text to lowercase
# Example: "GREAT Flight" becomes "great flight"

df['text'] = df['text'].str.lower()

/tmp/ipykernel_435/53610882.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['text'] = df['text'].str.lower()


In [107]:
# Function to remove punctuation
# Example: "great!" becomes "great"

def remove_punctuation(text):
    return text.translate(
        str.maketrans('', '', string.punctuation)
    )

df['text'] = df['text'].apply(remove_punctuation)

/tmp/ipykernel_435/158140673.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['text'] = df['text'].apply(remove_punctuation)


In [108]:
# Remove numbers from the text

def remove_numbers(text):
    new_text = ""

    for letter in text:
        if not letter.isdigit():
            new_text = new_text + letter

    return new_text

df['text'] = df['text'].apply(remove_numbers)

/tmp/ipykernel_435/1391611482.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['text'] = df['text'].apply(remove_numbers)


In [109]:
# Remove website links from the tweets

def remove_links(text):
    return re.sub(r'http\S+', '', text)

df['text'] = df['text'].apply(remove_links)

/tmp/ipykernel_435/426230574.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['text'] = df['text'].apply(remove_links)


In [110]:
# Import stopwords from NLTK

import nltk

from nltk.corpus import stopwords

In [111]:
# Download the English stopword list

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [112]:
# Stopwords are common words like:
# "the", "is", "and", "a"

stop_words = set(stopwords.words('english'))

In [113]:
# Remove common stopwords from each sentence

def remove_stopwords(text):

    words = text.split()

    cleaned_words = []

    for word in words:
        if word not in stop_words:
            cleaned_words.append(word)

    return ' '.join(cleaned_words)


df['text'] = df['text'].apply(remove_stopwords)

/tmp/ipykernel_435/1367998004.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['text'] = df['text'].apply(remove_stopwords)


In [114]:
# Look at the cleaned text

df.head()

,text,airline_sentiment
0,virginamerica dhepburn said,neutral
1,virginamerica plus youve added commercials exp...,positive
2,virginamerica didnt today must mean need take ...,neutral
3,virginamerica really aggressive blast obnoxiou...,negative
4,virginamerica really big bad thing,negative


In [115]:
# Import PorterStemmer
# Stemming changes words to a simpler root form
# Example: "delayed", "delaying" -> similar root

from nltk.stem import PorterStemmer

stemmer = PorterStemmer()

In [116]:
# Stem each word in the text

def stem_text(text):

    words = text.split()

    stemmed_words = []

    for word in words:
        stemmed_words.append(stemmer.stem(word))

    return ' '.join(stemmed_words)


df['text'] = df['text'].apply(stem_text)

/tmp/ipykernel_435/3782396589.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['text'] = df['text'].apply(stem_text)


In [117]:
# See the different sentiment classes

df['airline_sentiment'].value_counts()

,count
airline_sentiment,
negative,9178
neutral,3099
positive,2363


In [118]:
# Split data into training and testing sets
# 80% training
# 20% testing

X_train, X_test, y_train, y_test = train_test_split(
    df['text'],
    df['airline_sentiment'],
    test_size=0.20,
    random_state=42
)

In [119]:
# TF-IDF converts text into numbers
# Machine learning models need numerical input

tfidf = TfidfVectorizer()

X_train_tfidf = tfidf.fit_transform(X_train)

X_test_tfidf = tfidf.transform(X_test)

In [120]:
# Create the Logistic Regression classification model

model = LogisticRegression(max_iter=1000)

In [121]:
# Train the model using the training data

model.fit(X_train_tfidf, y_train)

LogisticRegression(max_iter=1000)

In [122]:
# Predict the sentiment of the test data

y_pred = model.predict(X_test_tfidf)

In [123]:
# Accuracy tells us how many predictions were correct

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

Accuracy: 0.7961065573770492


In [124]:
# F1-score measures classification performance
# It considers both precision and recall

f1 = f1_score(
    y_test,
    y_pred,
    average='weighted'
)

print("F1-score:", f1)

F1-score: 0.7849200391499821


In [125]:
# Get all the feature words created by TF-IDF

words = tfidf.get_feature_names_out()

In [126]:
# Show the 10 most important words for each sentiment class

words = tfidf.get_feature_names_out()

for i, sentiment in enumerate(model.classes_):

    top_indices = model.coef_[i].argsort()[-10:][::-1]

    print("\nTop 10 words for", sentiment)

    for index in top_indices:
        print(words[index])


Top 10 words for negative
hour
delay
worst
hr
cancel
bag
hold
fail
luggag
lost

Top 10 words for neutral
southwestair
jetblu
americanair
unit
hi
carri
virginamerica
atlanta
dal
saw

Top 10 words for positive
thank
great
awesom
love
amaz
best
appreci
excel
thx
kudo
